#  Expansión y Contracción en el Transformer: La Intuición tras la Subcapa MLP

Antes de profundizar en **Multi-Head Attention** y en técnicas de regularización como *Dropout*, es fundamental entender la subcapa **MLP (Multi-Layer Perceptron)** del bloque Transformer y el motivo matemático detrás de la técnica de **expansión y contracción dimensional**.

---

## 1. La Estructura del Transformer Block y el Flujo Residual

Un bloque Transformer estándar se compone de dos subcapas con una arquitectura simétrica:

1. **Subcapa de Atención (*Self-Attention Sublayer*):** Comprende la normalización previa (`LayerNorm`), el cálculo de atención y la suma al flujo residual original.
2. **Subcapa MLP (*Feed-Forward Sublayer*):** Comprende su propia `LayerNorm`, la red densa no lineal de dos pasos y la suma al flujo residual.

En ambas subcapas se extrae una copia del vector antes de la normalización para reincorporarla al final como una **conexión residual** ($X_{\text{out}} = X + \text{Sublayer}(\text{LayerNorm}(X))$).

---

## 2. El Flujo Interno de la Subcapa MLP: Expansión $4\times$ y Contracción

La subcapa MLP procesa la salida generada por la atención a través del siguiente esquema secuencial:

$$W_1 \longrightarrow \text{GELU} \longrightarrow W_2$$

* **Entrada:** Vector proveniente de la subcapa de atención con dimensión de embedding $d_{\text{model}}$ (por ejemplo, $d_{\text{model}} = 768$ en GPT-2 Small).
* **Expansión ($W_1$):** La primera proyección lineal expande típicamente la dimensionalidad **4 veces**:
  $$d_{\text{mlp}} = 4 \times d_{\text{model}} = 4 \times 768 = 3072$$
* **Activación No Lineal ($\text{GELU}$):** Aplica la no linealidad elemento por elemento en ese espacio de alta dimensión.
* **Contracción ($W_2$):** La segunda matriz proyecta el tensor de vuelta de $3072$ a la dimensión original de $768$.
* **Retorno al Residuo:** El resultado final se suma directamente al flujo residual principal.

---

## 3. ¿Por qué Expandir y Contraer? El Principio de Separabilidad Lineal

Esta técnica permite **linealizar relaciones complejas** que en su dimensión original no son separables:

* **El problema en baja dimensión ($\mathbb{R}^2$):** Si tenemos datos distribuidos en círculos concéntricos en un plano 2D, ningún hiperplano (línea recta) puede separar ambas clases; un clasificador puramente lineal no superaría el azar.
* **El truco de la dimensión superior ($\mathbb{R}^3$):** Al proyectar los datos a una tercera dimensión mediante una función no lineal (por ejemplo, $z = x^2 + y^2$, la distancia al origen), los círculos concéntricos se convierten en un paraboloide 3D. En ese espacio extendido, **un simple plano horizontal separa perfectamente ambas clases de forma lineal**.
* **Aplicación en Transformers:** La matriz $W_1$ y la función $\text{GELU}$ proyectan las activaciones a un espacio latente de $3072$ dimensiones donde los patrones, conceptos semánticos y reglas abstractas se vuelven linealmente separables y procesables para $W_2$.

---

## 4. Roles Complementarios: Atención vs. MLP

* **Subcapa de Atención (Intercambio entre Tokens / *Cross-Token Mixing*):**  
  Es la **única fase con comunicación contextual y temporal**. Utiliza $Q$, $K$ y $V$ con la máscara causal para que cada token recopile y filtre selectivamente información relevante de tokens pasados.
* **Subcapa MLP (Procesamiento por Token / *Position-Wise Processing*):**  
  **No mezcla información entre tokens**. Opera de forma aislada e idéntica en cada posición temporal ($1 \times 1$ conv / proyección independiente). Su función es procesar, refinar y almacenar conocimiento factual sobre las características ya mezcladas, ajustando los vectores de embedding para proyectar con mayor precisión el siguiente token.